In [ ]:
# Copyright (c) TorchGeo Contributors. All rights reserved.
# Licensed under the MIT License.

# Evaluating SSL Checkpoints

_Written by: Caleb Robinson_

Self-supervised learning (SSL) has no labels, so a training loss tells you very little about whether a run worked. The losses of different SSL tasks are on different scales and measure different things, and a loss that falls steadily is entirely compatible with a representation that has *collapsed* to a single point. None of TorchGeo's SSL tasks implement a meaningful `validation_step`, so there is no in-training metric to watch either.

The standard way to find out whether an SSL encoder learned anything is to freeze it, extract features, and fit a cheap classifier on top. A k-nearest-neighbors (kNN) probe is a good choice because it has no optimizer, no learning rate, and no regularization of its own, so it measures the representation rather than the tuning of the probe.

In this tutorial, we will:

- Pretrain a `SimCLR` task on **EuroSAT100** for a couple of epochs, so that we have a checkpoint to work with.
- Recover the frozen encoder from that checkpoint, which lives at a different attribute for each SSL task.
- Extract fixed-length features, being careful to reproduce the preprocessing used during training.
- Score those features with a kNN probe.
- Check the embeddings for representation collapse, which accuracy alone can hide.

This is the same procedure used for the numbers in the [SSL benchmark](../user/ssl_benchmark.rst), just on a much smaller dataset so it runs quickly.

## Setup

First, we install TorchGeo and scikit-learn.

In [ ]:
# On Colab, this ensures the latest TorchGeo is available.

!uv pip install torchgeo scikit-learn

## Imports

Next, we import TorchGeo and any other libraries we need.

In [ ]:
import os
import tempfile

import lightning.pytorch as pl
import numpy as np
import timm
import torch
import torch.nn.functional as F
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from torch import Tensor, nn

from torchgeo.datamodules import EuroSAT100DataModule
from torchgeo.datasets import EuroSAT100
from torchgeo.tasks import SimCLR

## Data

We use **EuroSAT100**, a 100-image subset of EuroSAT intended for demonstrations. It keeps the same classes and train/val/test structure as the full dataset, with 60 training and 20 validation images.

`EuroSAT100DataModule` standardizes each of the 13 Sentinel-2 bands using the dataset's per-band mean and standard deviation. That normalization matters later: we have to apply exactly the same transformation at evaluation time.

In [ ]:
root = os.path.join(tempfile.gettempdir(), 'eurosat100')
EuroSAT100(root=root, split='train', download=True)
EuroSAT100(root=root, split='val', download=True)

datamodule = EuroSAT100DataModule(root=root, batch_size=8, num_workers=0)
datamodule.setup('fit')

print(f'{len(datamodule.train_dataset)} train, {len(datamodule.val_dataset)} val images')

## Pretraining a task

To have something to evaluate, we pretrain a `SimCLR` task for a couple of epochs. SimCLR builds two augmented views of each image and pulls them together in feature space while pushing apart views of different images, so it never sees a label.

Two settings are worth pointing out:

- `in_channels=13` because EuroSAT is multispectral, not RGB.
- `size=64` keeps the augmented crops at the native EuroSAT resolution so this tutorial runs quickly on a CPU. The benchmark uses `size=224`, which scores considerably better but takes far longer.

A real pretraining run would use hundreds of epochs on tens of thousands of images. Two epochs on sixty images is only enough to show the mechanics, and it keeps this notebook fast enough to run on a CPU.

In [ ]:
pl.seed_everything(0)

task = SimCLR(
    model='resnet18',
    in_channels=13,
    version=2,
    lr=0.5,
    size=64,
    memory_bank_size=0,
)

trainer = pl.Trainer(
    accelerator='auto',
    devices=1,
    max_epochs=2,
    # Every SSL task's validation_step is a no-op, so there is nothing to run.
    limit_val_batches=0,
    num_sanity_val_steps=0,
    enable_checkpointing=False,
    logger=False,
)
trainer.fit(model=task, datamodule=datamodule)

checkpoint_path = os.path.join(tempfile.gettempdir(), 'simclr_eurosat100.ckpt')
trainer.save_checkpoint(checkpoint_path)

## Recovering the encoder from a checkpoint

Everything from here on treats the checkpoint as the only input, which is what you would do when scoring a run that finished hours ago.

An SSL checkpoint contains far more than the encoder: projection heads, predictor heads, and for MoCo an entire momentum copy of the backbone. We only want the encoder, and **where it lives depends on the task**:

| Task | State dict prefix |
| --- | --- |
| `SimCLR` | `backbone.` |
| `MoCo` | `backbone.` (note: `backbone_momentum.` is the momentum copy, not what we want) |
| `BYOL` | `model.backbone.model.` (nested inside a wrapper) |

Rather than reconstructing the whole task, we rebuild a bare [timm](https://huggingface.co/docs/timm/index) model using the hyperparameters saved in the checkpoint and load only those weights. Loading with `strict=True` is deliberate: a silent mismatch here would leave part of the encoder randomly initialized, and we would happily report the resulting number.

In [ ]:
def load_backbone(path: str) -> nn.Module:
    """Rebuild the frozen encoder stored in an SSL checkpoint."""
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    hparams = checkpoint['hyper_parameters']
    state_dict = checkpoint['state_dict']

    # BYOL nests the encoder inside its BackboneWrapper; SimCLR and MoCo do not.
    for prefix in ('model.backbone.model.', 'backbone.'):
        if any(key.startswith(prefix) for key in state_dict):
            break

    weights = {
        key[len(prefix) :]: value
        for key, value in state_dict.items()
        if key.startswith(prefix)
    }

    backbone = timm.create_model(
        hparams['model'], in_chans=hparams['in_channels'], num_classes=0
    )
    backbone.load_state_dict(weights, strict=True)
    return backbone.eval()


backbone = load_backbone(checkpoint_path)
print(f'recovered {sum(p.numel() for p in backbone.parameters()) / 1e6:.1f}M parameters')

## Extracting frozen features

The one thing that is easy to get wrong here is preprocessing. During training, the datamodule normalizes each batch inside `on_after_batch_transfer`, and *then* the task resizes it as part of its augmentation pipeline. That hook only runs inside a Lightning loop, so when we iterate a dataloader ourselves we have to apply `datamodule.aug` by hand, in the same order.

We also drop the random augmentation: features are extracted from unaugmented images, resized to whatever resolution the encoder was trained at.

Note that we do *not* reuse `datamodule.train_dataloader()` here. A training dataloader shuffles and, in TorchGeo, drops the last incomplete batch, which would silently discard a few images and misalign nothing but quietly shrink the probe's training set. For evaluation we want every sample, in a fixed order, so we build plain dataloaders over the underlying datasets.

`forward_head(forward_features(x), pre_logits=True)` asks timm for the pooled features that sit immediately before the classification layer, which works for both convolutional and transformer encoders.

In [ ]:
@torch.no_grad()
def extract_features(
    backbone: nn.Module, dataset: torch.utils.data.Dataset, size: int = 64
) -> tuple[np.ndarray, np.ndarray]:
    """Extract frozen features and labels for every sample in a dataset."""
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=False)
    features, labels = [], []
    for batch in dataloader:
        x: Tensor = batch['image']
        # Same order as training: standardize first, then resize.
        x = datamodule.aug({'image': x})['image']
        x = F.interpolate(x, size=(size, size), mode='bilinear', align_corners=False)

        z = backbone.forward_head(backbone.forward_features(x), pre_logits=True)
        features.append(z.flatten(1))
        labels.append(batch['label'])

    return torch.cat(features).numpy(), torch.cat(labels).numpy()


x_train, y_train = extract_features(backbone, datamodule.train_dataset)
x_val, y_val = extract_features(backbone, datamodule.val_dataset)

print(f'train features {x_train.shape}, val features {x_val.shape}')

## Fitting the kNN probe

Now we fit a 5-nearest-neighbors classifier on the training features and score it on the validation features.

We fit the probe twice, once on the raw features and once on standardized features, and report whichever is better. Some encoders produce features whose dimensions differ wildly in scale, which a Euclidean distance is sensitive to, so this keeps the comparison between encoders fair.

In [ ]:
def knn_score(
    x_train: np.ndarray, y_train: np.ndarray, x_eval: np.ndarray, y_eval: np.ndarray
) -> dict[str, float]:
    """Score frozen features with a kNN probe, raw and standardized."""
    scores = {}
    for name in ('raw', 'standardized'):
        if name == 'raw':
            a, b = x_train, x_eval
        else:
            scaler = StandardScaler()
            a, b = scaler.fit_transform(x_train), scaler.transform(x_eval)
        probe = KNeighborsClassifier(n_neighbors=5).fit(a, y_train)
        scores[name] = float(probe.score(b, y_eval))
    scores['best'] = max(scores['raw'], scores['standardized'])
    return scores


scores = knn_score(x_train, y_train, x_val, y_val)
for name, value in scores.items():
    print(f'{name:>13}: {value:.4f}')

How good is that number? On its own, it is meaningless. It only becomes informative next to a floor, and the cheapest useful floor is the *same encoder without any training*. If pretraining cannot beat a randomly initialized network, it has not learned anything.

Treat the gap below as a sanity check on the pipeline rather than as evidence that SimCLR works. With 20 validation images a single image is worth 0.05 accuracy, and two epochs on 60 images is nowhere near convergence. The [SSL benchmark](../user/ssl_benchmark.rst) runs this same probe properly and separates SimCLR from random initialization by roughly 0.07 on the full dataset.

In [ ]:
random_backbone = timm.create_model('resnet18', in_chans=13, num_classes=0).eval()
x_train_random, _ = extract_features(random_backbone, datamodule.train_dataset)
x_val_random, _ = extract_features(random_backbone, datamodule.val_dataset)

random_scores = knn_score(x_train_random, y_train, x_val_random, y_val)
print(f'   pretrained: {scores["best"]:.4f}')
print(f'random init.: {random_scores["best"]:.4f}')

## Checking for representation collapse

Accuracy can hide a failure mode that is specific to SSL. If a task collapses, every image maps to nearly the same vector. Distances between points become meaningless, but the training loss can look perfectly healthy while it happens, because many SSL losses are minimized by a constant output.

Two cheap diagnostics catch it:

- The **standard deviation** of the L2-normalized embeddings. A collapsed encoder drives this toward zero.
- The **mean pairwise cosine similarity** between embeddings. A collapsed encoder drives this toward one.

Report these alongside any accuracy number. In the benchmark sweep, a MoCo run reached a standard deviation of 0.0005 and dropped below the random-init floor while its loss barely moved.

In [ ]:
def collapse_diagnostics(features: np.ndarray) -> dict[str, float]:
    """Measure how degenerate a representation is."""
    normalized = F.normalize(torch.from_numpy(features).float(), dim=1)
    similarity = normalized @ normalized.T
    n = len(normalized)
    # Exclude the diagonal, which is always 1.
    off_diagonal = (similarity.sum() - n) / (n * (n - 1))
    return {
        'embedding_std': float(normalized.std(dim=0).mean()),
        'mean_pairwise_cosine': float(off_diagonal),
    }


for name, features in (('pretrained', x_train), ('random init.', x_train_random)):
    stats = collapse_diagnostics(features)
    print(
        f'{name:>12}: std={stats["embedding_std"]:.4f} '
        f'cosine={stats["mean_pairwise_cosine"]:.4f}'
    )

## Next steps

Two epochs on 60 images is not a serious pretraining run, so treat the numbers above as a demonstration of the mechanics rather than a result.

To evaluate a real run, apply the same procedure with the full protocol described in the [SSL benchmark](../user/ssl_benchmark.rst): pretrain on the full EuroSAT dataset at 224x224, sweep several learning rates, select the best on the validation split, and only then read the test split. Ready-to-run configurations for `SimCLR`, `MoCo`, and `BYOL`, each using the learning rate selected for it, are in [configs/ssl_benchmarking](https://github.com/torchgeo/torchgeo/tree/main/configs/ssl_benchmarking).